# example / pyfunc / linear

**Intent**
- Estimates `pyfunc_out` from `pyfunc_in`, and must:
  - Scale the input by a fitted gain plus a fitted offset.
  - Show a fitted model end to end, with every number reproducible.
  - Hold the rmse on `validation` and `backtest` to 0.5.
  - Stay a fixture, tested by `fab unittest` and `fab systest` and never released.

**Data**
- `training` is ten labelled points on a line with fixed noise, the only data the fit sees.
- `validation` is five more points, and `backtest` is ten further out, held back for acceptance.
- `golden` is six exactly representable inputs, then a request with the input missing that must give a signature error.

## Setup ([6.7.1](https://github.com/ggear/asystem/blob/master/src/rue/mlmodels/src/build/resources/doc/model.md#671-setup))

In [ ]:
import numpy as np

from mlmodels import setup_workspace

ws = setup_workspace("example/pyfunc/linear")

ws.outputs[0] = ws.outputs[0]

## Collect ([6.7.2](https://github.com/ggear/asystem/blob/master/src/rue/mlmodels/src/build/resources/doc/model.md#672-collect))

In [ ]:
from mlmodels import collect_run

NOISE = [0.04, -0.03, 0.02, -0.05, 0.01, 0.03, -0.02, 0.05, -0.04, 0.0]


def labelled(first, count):
    xs = [float(x) for x in range(first, first + count)]
    return {ws.inputs[0]: xs, ws.outputs[0]: [round(2.5 * x + 1.0 + NOISE[int(x) % 10], 4) for x in xs]}


collect_run(ws, 1000, label="Ten labelled points on a line with fixed noise", name="pyfunc_linear_training", source="builder", build=lambda: labelled(0, 10))
collect_run(ws, 1000, label="Five more points on the line", name="pyfunc_linear_validation", source="builder", build=lambda: labelled(10, 5))
collect_run(ws, 1000, label="Ten further points on the line held back for acceptance", name="pyfunc_linear_backtest", source="builder", build=lambda: labelled(15, 10))

## Prepare ([6.7.3](https://github.com/ggear/asystem/blob/master/src/rue/mlmodels/src/build/resources/doc/model.md#673-prepare))

In [ ]:
from mlmodels import prepare_split

prepare_split(ws, training=ws.run(1000, "pyfunc_linear_training"), validation=ws.run(1000, "pyfunc_linear_validation"), backtest=ws.run(1000, "pyfunc_linear_backtest"))
print("training", ws.training.height, "validation", ws.validation.height, "backtest", ws.backtest.height)

## Train ([6.7.5](https://github.com/ggear/asystem/blob/master/src/rue/mlmodels/src/build/resources/doc/model.md#675-train))

In [ ]:
gain, offset = np.polyfit(ws.training[ws.inputs[0]].to_numpy(), ws.training[ws.outputs[0]].to_numpy(), 1)
GAIN, OFFSET = round(float(gain), 1), round(float(offset), 1)
ws.params = {"gain": GAIN, "offset": OFFSET}
print("fitted gain", GAIN, "offset", OFFSET)
assert (GAIN, OFFSET) == (2.5, 1.0), "the fit did not recover the line that made the data"

## Define ([6.7.6](https://github.com/ggear/asystem/blob/master/src/rue/mlmodels/src/build/resources/doc/model.md#676-define))

In [ ]:
from mlmodels import define_model

GAIN, OFFSET = ws.params["gain"], ws.params["offset"]


def linear(pyfunc_in):
    return GAIN * pyfunc_in + OFFSET


define_model(ws, model=1000, flavour="pyfunc", source=linear, label="Least squares line, gain and offset built in", constants={"GAIN": GAIN, "OFFSET": OFFSET})

## Evaluate ([6.7.7](https://github.com/ggear/asystem/blob/master/src/rue/mlmodels/src/build/resources/doc/model.md#677-evaluate))

In [ ]:
from mlmodels import evaluate_gate, evaluate_metrics, evaluate_model, evaluate_models


def acceptance_checks():
    checks = []
    for split in ("validation", "backtest"):
        predicted = evaluate_model(ws, 1000, split)[ws.outputs[0]]
        checks.append(
            {
                "split": split,
                "gate": split == "backtest",
                "output": ws.outputs[0],
                "name": "rmse",
                "value": evaluate_metrics(getattr(ws, split)[ws.outputs[0]], predicted)["rmse"],
                "limit": 0.5,
                "direction": "max",
            }
        )
    return checks


evaluate_gate(ws, 1000, acceptance_checks)
evaluate_models(ws)

## Package ([6.7.8](https://github.com/ggear/asystem/blob/master/src/rue/mlmodels/src/build/resources/doc/model.md#678-package))

In [ ]:
from mlmodels import package_model

xs = [-1.0, 0.0, 0.25, 1.0, 2.5, 10.0]
golden = {ws.inputs[0]: [*xs, None], "expect": ["ok"] * len(xs) + ["error"]}
package_model(ws, 1000, golden, target={"kind": "embedded", "language": "python"})